# Agentic design — a controlled AI campaign

In notebook 07 the AI proposes **one** change and you decide. Here it runs a **campaign**: it proposes
parameter values, Vegeta builds each candidate as a new revision, runs the FEA, checks the criteria,
shows the model the full table of results, and asks for the next move. It keeps going until a limit
is reached.

```
      ┌──────────── table of every candidate so far ◄────────────┐
      ▼                                                          │
 proposer ──► checks (free params, ranges, bounds, no repeats) ──► approval policy
 (Claude)                                                          │ yes
                                                                   ▼
                                    branch revision ─► generate ─► FEA ─► criteria ─► record
```

Where the authority sits:
- **You** write the loads, material, criteria, objective, the free parameters and the budget.
- **The AI** can only choose parameter values. It cannot change the design source, the loads or the material.
- **Vegeta** refuses invalid proposals, and nothing runs until your approval policy says yes.
- **Every candidate** is an ordinary immutable revision with its own evaluation.
- **The campaign never labels a design.** You choose, at the end.

Limits: `max_iterations`, `max_tokens`, `max_minutes`, `patience` (stop after this many steps with no
improvement), and a file named `STOP` in the campaign directory.

In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd
from vegeta import core, talos
from vegeta.ai import Analysis, Budget, Campaign, Criterion, Objective, ClaudeConfig, ClaudeProposer
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz

WS = Path("_runs/agentic"); shutil.rmtree(WS, ignore_errors=True)
ws = core.Workspace.create(WS, name="agentic bracket")
bracket = ws.add_design("bracket", "vegeta.dedalus.examples:Bracket")
pd.DataFrame(bracket.record["parameters"]).set_index("name")

## 1. The engineering problem (written by you)

An 80 mm aluminium bracket, clamped at one end, with 150 N down on the other end.

**Requirements:**
- safety factor against yield ≥ 2.5;
- tip deflection ≤ 0.8 mm;
- among the designs that meet both, the lowest volume (mass) wins.

**Free parameters:** `thickness` and `width`. `length` and `hole_diameter` are fixed, because they are
set by the parts the bracket bolts to. A company rule adds bounds on top of the design's own ranges:
thickness between 3 and 12 mm, width between 25 and 60 mm.

In [ ]:
ALU = talos.Material("Al 6061-T6", youngs_modulus=68900, poissons_ratio=0.33, yield_strength=276,
                     source="nominal handbook values")

def cantilever(rev):
    L = rev.params["length"]
    return talos.StructuralModel(
        rev.step, "mm-N-MPa", ALU,
        regions=[talos.SurfacesOnPlane("clamped", "x", -L / 2), talos.SurfacesOnPlane("loaded", "x", L / 2)],
        supports=[talos.FixedSupport("clamped")], loads=[talos.Force("loaded", fz=-150.0)],
        mesh_settings=talos.MeshSettings(element_size=3.0), name="cantilever")

ANALYSES  = [Analysis("fea", "cantilever", cantilever)]
CRITERIA  = [Criterion("fea.cantilever.safety_factor_yield", ">=", 2.5),
             Criterion("fea.cantilever.max_displacement", "<=", 0.8)]
OBJECTIVE = Objective("geometry.volume", "min")
FREE      = ["thickness", "width"]
BOUNDS    = {"thickness": (3.0, 12.0), "width": (25.0, 60.0)}

start = bracket.new_revision(thickness=4.0, width=40.0, note="baseline for the campaign")
start

## 2. The proposer

If `ANTHROPIC_API_KEY` is set, Claude proposes (`effort="medium"` is enough for a parameter search, and
cheaper). Without a key, the notebook uses a small **scripted** proposer so that it still runs. The
scripted proposer is not an AI: it raises the thickness while the design fails, and once a design passes
it tries thinner and narrower versions of it. Both see the same context. Print `campaign.context()["notes"]`
below to see exactly what the model reads.

In [ ]:
class ScriptedProposer:
    # Deterministic stand-in for the AI: a simple coordinate search over thickness and width.
    name = "scripted"

    def describe(self):
        return {"provider": "none", "model": "scripted-coordinate-search"}

    def propose(self, context, instruction, history):
        c = context["campaign"]
        values = {p["name"]: p["value"] for p in context["parameters"]}
        tried = [it.get("parameters") for it in c["iterations"] if it.get("parameters")]
        inc = next(it for it in c["iterations"] if it.get("revision") == c["incumbent"])
        if not inc["feasible"]:
            move = {"thickness": round(values["thickness"] * 1.25 * 2) / 2}
            why = "incumbent fails the criteria: +25 % thickness (bending stiffness ~ t^3)"
        else:
            move = None
            for cand, reason in (({"thickness": values["thickness"] - 0.5}, "thinner by 0.5 mm"),
                                 ({"width": values["width"] - 5.0}, "narrower by 5 mm")):
                if {**values, **cand} not in tried:
                    move, why = cand, f"incumbent is feasible: try {reason} to save volume"
                    break
            if move is None:
                return ({"kind": "answer", "summary": "no untried neighbour of the best design",
                         "rationale": "both neighbouring moves of the incumbent were evaluated",
                         "source": None, "parameters": {}, "expected_effects": [], "risks": []},
                        {"input_tokens": 0, "output_tokens": 0})
        return ({"kind": "parameters", "summary": str(move), "rationale": why, "source": None,
                 "parameters": move, "expected_effects": [], "risks": []},
                {"input_tokens": 0, "output_tokens": 0})

if os.environ.get("ANTHROPIC_API_KEY"):
    proposer = ClaudeProposer(ClaudeConfig(model="claude-opus-5", effort="medium",
                                           extra_system="Aluminium 6061 bracket, machined from plate; thickness in 0.5 mm steps."))
else:
    proposer = ScriptedProposer()
proposer.describe()

## 3. The approval policy (also written by you)

`approval="ask"` asks you `y/N` before every candidate. `"auto"` runs every valid proposal, and you have
to opt into it explicitly. In between, a **written policy** is a plain function. The one below approves
a step only if each change stays within ±30 % of the current best design (no wild jumps) and prints
every decision. If the policy says no, the campaign **stops**; it does not ask the AI for something else.

In [ ]:
def small_steps_only(p):
    base = ws.revision(p["base"]).params
    ok = all(abs(v - base[k]) <= 0.3 * abs(base[k]) for k, v in p["changes"].items())
    print(f"  approval: step {p['step']} {p['changes']} from {p['base']} -> {'YES' if ok else 'NO (step too large)'}")
    return ok

In [ ]:
campaign = Campaign(
    ws, start, analyses=ANALYSES, criteria=CRITERIA, objective=OBJECTIVE, proposer=proposer,
    free=FREE, bounds=BOUNDS, approval=small_steps_only,
    budget=Budget(max_iterations=8, max_tokens=200_000, max_minutes=20, patience=3),
    goal="lightest bracket that carries 150 N at the tip with SF >= 2.5 and <= 0.8 mm deflection",
    name="light_bracket")
campaign

## 4. Run it

This is the only cell that does work. To stop a running campaign cleanly, create
`_runs/agentic/campaigns/light_bracket/STOP`; the loop finishes the current candidate and stops.

In [ ]:
campaign.run()

## 5. What happened

In [ ]:
campaign.table()

In [ ]:
fig = campaign.plot()

In [ ]:
ws.status()

In [ ]:
best = campaign.best
print("best feasible revision:", best)
print("stopped because:", campaign.stopped, "| tokens used:", campaign.tokens)

def fea_files(rev):
    d = rev.evaluation("fea", "cantilever").directory
    return d / "model.frd", d / "mesh.msh"

if best is not None:
    frd, msh = fea_files(best)
    tviz.show(tviz.plot_results(frd, mesh=msh, field="von_mises", clim=(0, 276 / 2.5)))

In [ ]:
frd0, msh0 = fea_files(start)          # the baseline, same colour scale: red = beyond the allowed stress
tviz.show(tviz.plot_results(frd0, mesh=msh0, field="von_mises", clim=(0, 276 / 2.5)))

In [ ]:
if best is not None:
    g0 = start.design.load().generate(**start.params)
    g1 = best.design.load().generate(**best.params)
    g0.name, g1.name = f"baseline {start.id}", f"best {best.id}"
    dviz.show(dviz.plot3d(g0, g1, opacity=0.6))
    fig = dviz.plot_sections(g1, normal="x", positions=[-20.0, 0.0, 20.0], cols=3)

## 6. Audit: what the model saw and what was recorded

The campaign directory contains `campaign.json` (the full state, rewritten after every step) and
`events.jsonl` (one line per step, including the proposer's rationale and token usage). Every
candidate's own files (STEP, STL, mesh, CalculiX deck and results, commands) are under
`revisions/<id>/`, as in any other revision.

In [ ]:
print(campaign.context()["notes"])

In [ ]:
import json
for line in (campaign.dir / "events.jsonl").read_text().splitlines():
    e = json.loads(line)
    print(e["event"], e.get("step", ""), e.get("revision") or "-", e.get("status", ""), "|",
          (e.get("rationale") or e.get("reason") or "")[:110])

## 7. Your decision

The campaign found a candidate. Accepting it is a separate, explicit act. Labelling it `preferred`
records that decision in the revision's annotations, next to the evidence.

In [ ]:
if best is not None:
    best.label("preferred", note=f"from campaign {campaign.name}; checked the stress plot and the table")
ws.status()

## 8. Continue, or change the problem

- **Continue:** build the `Campaign` again with the same `name` and a new `Budget`, then call `run()`.
  It picks up from `campaign.json`, and every earlier step is still in the table the model sees.
- **Change the problem** (a new load, criterion or free parameter): start a **new** campaign with a new
  name. Results obtained under different requirements must not be mixed.
- **Change the design itself** (for example, add ribs): that is a source change, so it goes through
  `DesignSession` (notebook 07), which shows you the diff before you accept. After that, run a new
  campaign on a revision that uses the new source.
- **Stop at any time** by creating `campaigns/<name>/STOP`.